# Mapping the Workforce: A Social Network Analysis of Job Interconnectivity
## O*NET to Gephi — Pearson Correlation Approach

**Edge Definition**: Two occupations are linked if the Pearson correlation between their skill importance score profiles exceeds 90% similarity
> **Intuition:** If two jobs rank their skills in a similar order of importance, they are connected.
> *"These occupations value the same skills with the same level of priority."*

### Required Files
Download from [O\*NET 30.1](https://www.onetcenter.org/database.html) as **Text/Tab-delimited** (**Note:** Data will be aggregated between these two data files):
- `Occupation Data.txt`
- `Skills.txt`

### Outputs
- `occ_nodes_random.csv` — Node table for Gephi with a set percentage of nodes randomly removed
- `occ_edges_random.csv` — Edge table for Gephi with a set percentage of edges randomly removed

### Tools
Google Gemini was used for error correction and code restructuring

### 1. Imports

In [31]:
import pandas as pd
import numpy as np
from itertools import combinations

### 2. Initialization

In [32]:
OCCUPATION_FILE = "Occupation Data.txt"
SKILLS_FILE     = "Skills.txt"
THRESHOLD       = 0.95   # Pearson correlation cutoff (-1 to 1)
SCALE           = "IM"   # Select only IMPORTANCE values

### 3. Load Data from Files

In [33]:
def load_data():
    occ = pd.read_csv(OCCUPATION_FILE, delimiter="\t", usecols=["O*NET-SOC Code", "Title"]) # Read data from occupation file
    skills = pd.read_csv(SKILLS_FILE, delimiter="\t",usecols=["O*NET-SOC Code", "Element ID", "Scale ID", "Data Value"]) # Read data from skills file
    return occ, skills

occ, skills = load_data()
occ.head() # Display the first 5 rows

,O*NET-SOC Code,Title
0,11-1011.00,Chief Executives
1,11-1011.03,Chief Sustainability Officers
2,11-1021.00,General and Operations Managers
3,11-1031.00,Legislators
4,11-2011.00,Advertising and Promotions Managers


### 4. Build Matrix

In [34]:
def build_matrix(skills):
    filtered = skills[skills["Scale ID"] == SCALE] # Filter skills where the Scale ID is only IMPORTANCE values

    matrix = filtered.pivot_table(
        index="O*NET-SOC Code",
        columns="Element ID",
        values="Data Value",
        aggfunc="mean"
    ).fillna(0) # Create a matrix by pivoting the skills array so that each skill SOC code and Element ID column has a data value.
    return matrix

matrix = build_matrix(skills) # Build the matrix by calling function
matrix.head() # Retrieve the top 5 entries

Element ID,2.A.1.a,2.A.1.b,2.A.1.c,2.A.1.d,2.A.1.e,2.A.1.f,2.A.2.a,2.A.2.b,2.A.2.c,2.A.2.d,...,2.B.3.k,2.B.3.l,2.B.3.m,2.B.4.e,2.B.4.g,2.B.4.h,2.B.5.a,2.B.5.b,2.B.5.c,2.B.5.d
O*NET-SOC Code,,,,,,,,,,,,,,,,,,,,,
11-1011.00,4.12,4.00,4.12,4.25,3.25,1.62,4.38,3.75,3.12,4.00,...,1.50,1.0,1.88,4.75,4.12,4.25,4.00,4.25,4.00,4.25
11-1011.03,4.00,4.00,4.12,4.00,2.88,2.12,4.12,3.75,3.38,3.75,...,1.00,1.0,1.88,3.88,3.88,3.88,3.38,2.88,2.25,3.12
11-1021.00,4.00,4.00,3.50,4.00,2.62,1.50,3.88,3.62,3.00,4.00,...,1.75,1.0,2.38,3.62,3.12,3.12,3.62,3.00,3.12,3.75
11-2011.00,3.75,4.12,3.75,4.00,3.00,1.62,4.00,3.25,3.00,3.25,...,1.00,1.0,1.62,3.75,3.12,3.12,3.50,2.75,2.62,3.12
11-2021.00,3.88,3.88,3.25,3.88,2.75,1.75,3.88,3.88,3.12,3.75,...,1.00,1.0,1.88,3.75,3.25,3.50,3.50,2.88,2.62,3.38


### 5. Compute Edges using Pearson Correlation

In [35]:
def compute_edges(matrix):

    corr_matrix = matrix.T.corr() # First transpose the skill matrix, then compute pearson correlation
    codes = corr_matrix.index.tolist() # Retrieve the columns of the matrix as a list
    edges = [] # Initialize an array for edges

    # Loop through the occupations and check every unique pair of occupations
    for i in range(len(codes)):
        for j in range(i + 1, len(codes)):
            # Retrieve the correlation value between occupation i and j
            corr = corr_matrix.iloc[i, j]

            # Only keep pairs whose pearson correlation is above or equal to set threshold
            if corr >= THRESHOLD:
                # Add the entry to the edge list when condition met
                edges.append({
                    "Source": codes[i],
                    "Target": codes[j],
                    "Weight": round(float(corr), 4)
                })

    edges_df = pd.DataFrame(edges) # Create a dataframe using the edges
    return edges_df

edges_df = compute_edges(matrix) # Compute the edges using the previously created matrix
edges_df.head() # Retrieve the top 5 entries

,Source,Target,Weight
0,11-1011.00,11-2022.00,0.9562
1,11-1011.00,11-3071.04,0.9584
2,11-1011.03,11-2011.00,0.9616
3,11-1011.03,11-2021.00,0.9576
4,11-1011.03,11-2022.00,0.9597


### 6. Nodes Table

In [36]:
def build_nodes(occ, edges_df):
    connected = set(edges_df["Source"]).union(set(edges_df["Target"])) # Collect all occupation that appears in the edge dataframe (use set to only collect unique occupations)
    nodes_df = occ[occ["O*NET-SOC Code"].isin(connected)].copy() # Filter occupations from the dataset to only keep the occupations that are within the edges dataframe
    nodes_df = nodes_df.rename(columns={"O*NET-SOC Code": "Id", "Title": "Label"}) # Rename columns as expected by Gephi

    nodes_df["SOC_Major"] = nodes_df["Id"].str[:2] # Extract the first two characters of the occupation code

    soc_labels = { # Dictionary of each occupation code and label (Retrieved from https://www.onetcenter.org/database.html)
        "11": "Management",
        "13": "Business & Financial",
        "15": "Computer & Math",
        "17": "Architecture & Engineering",
        "19": "Life, Physical & Social Science",
        "21": "Community & Social Service",
        "23": "Legal",
        "25": "Education",
        "27": "Arts & Media",
        "29": "Healthcare Practitioners",
        "31": "Healthcare Support",
        "33": "Protective Service",
        "35": "Food Preparation",
        "37": "Building & Grounds",
        "39": "Personal Care",
        "41": "Sales",
        "43": "Office & Admin Support",
        "45": "Farming, Fishing & Forestry",
        "47": "Construction & Extraction",
        "49": "Installation & Repair",
        "51": "Production",
        "53": "Transportation",
        "55": "Military"
    }
    nodes_df["SOC_Group"] = nodes_df["SOC_Major"].map(soc_labels).fillna("Other") # Map each SOC label to each occupation
    return nodes_df # Return the nodes dataframe

nodes_df = build_nodes(occ, edges_df) # Build the nodes
nodes_df.head() # Retrieve the top 5 entries

,Id,Label,SOC_Major,SOC_Group
0,11-1011.00,Chief Executives,11,Management
1,11-1011.03,Chief Sustainability Officers,11,Management
2,11-1021.00,General and Operations Managers,11,Management
4,11-2011.00,Advertising and Promotions Managers,11,Management
5,11-2021.00,Marketing Managers,11,Management


### 7. Randomly Remove Percentage of Nodes

In [37]:
REMOVE_PERCENT = 0.20 

def remove_random_nodes(nodes, edges, percent):
    """
    Randomly removes a percentage of nodes and cleans up 
    the edges associated with them.
    """
    # Calculate how many nodes to keep
    keep_fraction = 1.0 - percent
    
    # pandas DataFrame.sample:
    # https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html
    nodes_sampled = nodes.sample(frac=keep_fraction, random_state=42)
    
    # Get the list of remaining IDs
    remaining_ids = set(nodes_sampled["Id"])
    
    # Only keep edges where both Source and Target still exist
    # pandas Series.isin:
    # https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html
    edges_filtered = edges[
        edges["Source"].isin(remaining_ids) & 
        edges["Target"].isin(remaining_ids)
    ].copy()
    
    print(f"Removed {percent*100}% of nodes.")
    print(f"Nodes remaining: {len(nodes_sampled)}")
    print(f"Edges remaining: {len(edges_filtered)}")
    
    return nodes_sampled, edges_filtered

nodes_df, edges_df = remove_random_nodes(nodes_df, edges_df, REMOVE_PERCENT)

Removed 20.0% of nodes.
Nodes remaining: 420
Edges remaining: 2011


### 8. Export CSV to Gephi

In [16]:
def export(nodes_df, edges_df):
    nodes_df.to_csv("occ_nodes_random.csv", index=False) # Convert the nodes list into a csv file
    edges_df.to_csv("occ_edges_random.csv", index=False) # Convert the edges list into a csv file

export(nodes_df, edges_df) # Export the nodes and edges list